# SCBE Code A/B Matched-Budget Benchmark

This notebook fixes the broken Kaggle CPU comparison by matching the baseline and triangulated code datasets to the same approximate token budget before training.

Target lane:
- Model: `Qwen/Qwen2.5-Coder-0.5B-Instruct`
- Budget: matched baseline vs triangulated token count
- Runtime goal on free Colab T4: roughly 20-45 minutes instead of a 12 hour CPU stall


In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate peft trl bitsandbytes huggingface_hub pytest


In [ ]:
# Clone the repo
import os, subprocess

REPO_DIR = "/content/SCBE-AETHERMOORE"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/issdandavis/SCBE-AETHERMOORE.git", REPO_DIR], check=True)
else:
    print("Repo already present")

%cd /content/SCBE-AETHERMOORE


In [ ]:
# Optional: login to Hugging Face if you want to push results later
from huggingface_hub import login
login()


In [ ]:
# Build matched-budget corpora locally
!python scripts/research/train_code_ab_fast.py --prepare-only


In [ ]:
# Inspect the matched-budget manifest
import json
from pathlib import Path

manifest_path = Path("artifacts/research/code_ab_fast/manifest.json")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print(json.dumps(manifest, indent=2))


In [ ]:
# Matched-budget A/B training on GPU
import json
import time
from pathlib import Path

import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer

BASE_MODEL = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
RUN_ROOT = Path("/content/code_ab_outputs")
RUN_ROOT.mkdir(parents=True, exist_ok=True)

def train_condition(data_path: str, run_name: str):
    print(f"\n=== Training {run_name} ===")
    ds = load_dataset("json", data_files=data_path, split="train")

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model = prepare_model_for_kbit_training(model)
    lora = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    model = get_peft_model(model, lora)

    def tokenize_fn(batch):
        return tokenizer(batch["text"], truncation=True, max_length=512, padding="max_length")

    tokenized = ds.map(tokenize_fn, batched=True, remove_columns=["text"])
    tokenized = tokenized.map(lambda batch: {"labels": batch["input_ids"]}, batched=True)

    args = TrainingArguments(
        output_dir=str(RUN_ROOT / run_name),
        num_train_epochs=1,
        max_steps=75,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        logging_steps=10,
        save_strategy="no",
        bf16=torch.cuda.is_available(),
        gradient_checkpointing=True,
        report_to="none",
    )

    trainer = Trainer(model=model, args=args, train_dataset=tokenized)
    t0 = time.time()
    trainer.train()
    elapsed = time.time() - t0

    final_loss = None
    for entry in reversed(trainer.state.log_history):
        if "loss" in entry:
            final_loss = float(entry["loss"])
            break

    result = {
        "run_name": run_name,
        "elapsed_seconds": round(elapsed, 2),
        "final_loss": final_loss,
        "rows": len(ds),
    }
    del model, trainer
    torch.cuda.empty_cache()
    return result

baseline_path = "artifacts/research/code_ab_fast/baseline_matched.jsonl"
triangulated_path = "artifacts/research/code_ab_fast/triangulated_matched.jsonl"

baseline_result = train_condition(baseline_path, "baseline_matched")
triangulated_result = train_condition(triangulated_path, "triangulated_matched")

summary = {
    "baseline": baseline_result,
    "triangulated": triangulated_result,
}
if baseline_result["final_loss"] is not None and triangulated_result["final_loss"] is not None:
    delta = triangulated_result["final_loss"] - baseline_result["final_loss"]
    summary["delta_loss"] = round(delta, 4)
    summary["relative_improvement_pct"] = round(abs(delta) / baseline_result["final_loss"] * 100, 2)

summary_path = RUN_ROOT / "code_ab_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))


If this notebook runs cleanly, use the summary file at `/content/code_ab_outputs/code_ab_summary.json` as the result packet. If triangulated still loses, that is still a valid result — the point of this lane is a fair benchmark, not forcing a win.
